# IAC Training on TA-RWARE — Local / CPU Demo

**Project:** Scalable MARL for Warehouse Logistics (Krnjaic et al., 2023 — arXiv 2212.11498v3)  
**Algorithm:** IAC — Independent Actor-Critic (paper Table I, GTP section)  
**Environment:** Small GTP — `tarware-medium-8agvs-4pickers-partialobs-v1`

> **This notebook runs a short smoke test (200 episodes) to verify the training pipeline works.**  
> For full training (3,000 episodes), use `iac_colab.ipynb` on Google Colab with a T4 GPU.

---

## Scope

| Setting | Local (this notebook) | Colab (iac_colab.ipynb) |
|---------|----------------------|-----------------------|
| Target device | CPU (or GPU if available) | T4 GPU |
| Episodes | 200 (smoke test) | 3,000 (full training) |
| Timesteps | 100,000 | 1,500,000 |
| Expected time | ~5–20 min on CPU | ~1–2 hours on T4 |
| Purpose | Verify pipeline | Full training run |

---

## IAC Algorithm

- 12 independent agents (8 AGVs + 4 pickers), each with their own Actor + Critic network
- Architecture: 2 FC layers × 64 units, ReLU activations, GAE training (γ=0.99)
- No weight sharing — each agent learns purely from its own experience
- This is the simplest algorithm from the paper, directly comparable to Table I

## Step 1: Install EPyMARL and TA-RWARE

Run this once. If packages are already installed, pip will skip them.

In [ ]:
import subprocess, sys, os

PROJECT_ROOT = os.path.abspath('..')
EPYMARL_DIR  = os.path.join(PROJECT_ROOT, 'epymarl')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'experiments', 'results', 'iac_local_results')

os.makedirs(RESULTS_DIR, exist_ok=True)

# Use sys.executable -m pip — avoids 'pip not found' on Windows
PIP = f'"{sys.executable}" -m pip'

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if result.returncode != 0 and result.stderr:
        print(f'  STDERR: {result.stderr[-1000:]}')
    else:
        last = result.stdout.strip().split('\n')[-1] if result.stdout.strip() else 'OK'
        print(f'  {last}')
    return result.returncode

print('=== Installing TA-RWARE (from local repo) ===')
run(f'{PIP} install -q -e "{PROJECT_ROOT}"')

if not os.path.exists(EPYMARL_DIR):
    print('\n=== Cloning EPyMARL ===')
    run(f'git clone --quiet https://github.com/uoe-agents/epymarl "{EPYMARL_DIR}"')
else:
    print('\n=== EPyMARL already cloned ===')

print('\n=== Installing EPyMARL dependencies ===')
run(f'{PIP} install -q -r requirements.txt', cwd=EPYMARL_DIR)

print('\n=== Installing extras ===')
run(f'{PIP} install -q pyastar2d sacred pymongo')

print(f'\nPython      : {sys.executable}')
print(f'EPyMARL dir : {EPYMARL_DIR}')
print(f'Results dir : {RESULTS_DIR}')
print('All installations complete!')

## Step 2: Verify Environment and Device

In [2]:
import torch
import gymnasium as gym
import tarware

print('=== Device ===')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('CPU only (no CUDA GPU detected)')
    print('Training will be slower — consider using iac_colab.ipynb for full runs.')

print('\n=== Environment ===')
ENV_ID = 'tarware-medium-8agvs-4pickers-partialobs-v1'
env = gym.make(ENV_ID)
obs, _ = env.reset(seed=42)

print(f'Env ID    : {ENV_ID}')
print(f'N agents  : {env.unwrapped.n_agents} (AGVs={env.unwrapped.num_agvs}, Pickers={env.unwrapped.num_pickers})')
print(f'Max steps : {env.unwrapped.max_steps}')

if hasattr(env.observation_space, 'spaces'):
    shapes = [sp.shape for sp in env.observation_space.spaces]
    print(f'Obs shapes: {set(str(s) for s in shapes)}')
if hasattr(env.action_space, 'spaces'):
    n_actions = [sp.n for sp in env.action_space.spaces]
    print(f'N actions : {set(n_actions)}')
env.close()
print('\nEnvironment OK!')

ModuleNotFoundError: No module named 'tarware'

## Step 3: Configure Training

200 episodes (100,000 timesteps) is enough to verify the pipeline and see initial learning.
Increase `T_MAX` if you have more time or a GPU available.

In [ ]:
# Training budget — adjust as needed
T_MAX = 100_000    # 200 episodes * 500 steps  (~5-20 min on CPU)
# T_MAX = 500_000  # 1000 episodes             (~30-60 min on CPU)
# T_MAX = 1_500_000 # 3000 episodes (Colab only, use iac_colab.ipynb)

SEED = 42
N_EPISODES_APPROX = T_MAX // 500

CONFIG_OVERRIDES = [
    f'env_args.key={ENV_ID}',
    'env_args.time_limit=500',
    'env_args.common_reward=False',
    f't_max={T_MAX}',
    f'seed={SEED}',
    'save_model=True',
    f'results_save_dir={RESULTS_DIR}',
    'use_tensorboard=False',
]

print('Training Configuration:')
print(f'  t_max     : {T_MAX:,} steps (~{N_EPISODES_APPROX} episodes)')
print(f'  seed      : {SEED}')
print(f'  results   : {RESULTS_DIR}')
print(f'  device    : {"GPU" if torch.cuda.is_available() else "CPU"}')
print('\nNote: For full training (3000 ep), use iac_colab.ipynb on Google Colab T4 GPU.')

## Step 4: Run IAC Training

Launches EPyMARL's IAC algorithm. Output is streamed in real-time.

In [ ]:
import subprocess, sys, time

cmd = [
    sys.executable, 'src/main.py',
    '--config=iac',
    '--env-config=gymma',
    'with',
] + CONFIG_OVERRIDES

print(f'Running EPyMARL from: {EPYMARL_DIR}')
print(f'Timesteps: {T_MAX:,} (~{T_MAX//500} episodes)')
print('=' * 70)

start_time = time.time()

try:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        cwd=EPYMARL_DIR,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print('\nTraining interrupted.')

elapsed = time.time() - start_time
print('\n' + '=' * 70)
print(f'Exit code : {proc.returncode}')
print(f'Time      : {elapsed:.0f}s ({elapsed/60:.1f} min)')
if proc.returncode == 0:
    print('Done!')
else:
    print('Training ended with errors — see output above for details.')

## Step 5: Load Results and Plot Learning Curve

In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Find latest Sacred run directory
run_dirs = sorted([d for d in glob.glob(os.path.join(RESULTS_DIR, '*/')) if os.path.isdir(d)])

if not run_dirs:
    print(f'No results in: {RESULTS_DIR}')
    print('Run Step 4 first.')
else:
    LATEST_RUN = run_dirs[-1]
    print(f'Loading: {LATEST_RUN}')

    with open(os.path.join(LATEST_RUN, 'metrics.json')) as f:
        metrics = json.load(f)

    if 'return_mean' not in metrics:
        print('return_mean not found. Available:', list(metrics.keys()))
    else:
        data = metrics['return_mean']['values']
        steps = np.array([v['steps'] for v in data])
        returns_raw = np.array([v['value'] for v in data])

        N_AGENTS = 12
        MAX_STEPS = 500
        pick_rate_approx = returns_raw * N_AGENTS * 3600 / (5 * MAX_STEPS)

        def smooth(v, w=10):
            return np.convolve(v, np.ones(w)/w, mode='valid') if len(v) >= w else v

        steps_sm = steps[len(steps) - len(smooth(returns_raw)):]
        pick_sm = smooth(pick_rate_approx)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(steps, pick_rate_approx, alpha=0.3, color='steelblue')
        ax.plot(steps_sm, pick_sm, color='steelblue', linewidth=2, label='IAC (ours, local)')
        ax.axhline(y=52.16, color='darkorange', linestyle='--', linewidth=2, label='CTA ours (52.16)')
        ax.axhline(y=52.7,  color='gray',       linestyle=':',  label='CTA paper (52.7)')
        ax.axhline(y=65.2,  color='green',      linestyle=':',  label='IAC paper 10k ep (65.2)')
        ax.axhline(y=66.7,  color='red',        linestyle=':',  label='HIAC paper 10k ep (66.7)')
        ax.set_xlabel('Training Timesteps')
        ax.set_ylabel('Approx. Pick Rate (order-lines / hr)')
        ax.set_title(f'IAC Training Curve — Local Smoke Test ({T_MAX//500} episodes)')
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plot_path = os.path.join(RESULTS_DIR, 'iac_local_curve.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        print(f'Plot saved: {plot_path}')
        plt.show()

        final_pick = float(pick_sm[-1]) if len(pick_sm) > 0 else None
        print(f'\nApprox. final pick rate: {final_pick:.2f}' if final_pick else 'N/A')
        print(f'(Expected: will be below CTA at only {T_MAX//500} episodes; learning is still early)')

## Step 6: Summary Table

In [ ]:
try:
    iac_val = final_pick
except NameError:
    iac_val = None

print('=' * 68)
print('PERFORMANCE COMPARISON — Small GTP')
print('=' * 68)
rows = [
    ('Random (our Exp 2)',              '9.63 +/- 0.12'),
    ('CTA heuristic (our Exp 1)',       '52.16 +/- 0.54'),
    ('CTA (paper Table I)',             '52.7 +/- 0.9'),
    (f'IAC ours ({T_MAX//500} ep, local)', f'~{iac_val:.1f} (approx)' if iac_val else '[see plot]'),
    ('IAC (paper, 10k ep)',             '65.2 +/- 0.5'),
    ('HIAC (paper, 10k ep)',            '66.7 +/- 0.3'),
]
for name, val in rows:
    print(f'  {name:<35} {val}')
print('=' * 68)
print()
print('Key takeaway:')
print('  At just 200 episodes, IAC is in early training and below CTA.')
print('  At 3000+ episodes (Colab), IAC should surpass CTA, showing')
print('  that neural RL improves over the rule-based heuristic.')
print()
print('To run full training: open iac_colab.ipynb on Google Colab (T4 GPU).')

---

## Troubleshooting

| Error | Fix |
|-------|-----|
| `ModuleNotFoundError: tarware` | Re-run Step 1 install cell |
| `No module named 'epymarl'` | Check `EPYMARL_DIR` path is correct |
| EPyMARL crashes on `env_args` | Try quoting: `'env_args.key=tarware-...'` |
| Very slow training | Normal on CPU; use Colab for speed |
| Sacred/MongoDB warnings | Safe to ignore; results still saved |

## Relationship to Our Other Experiments

This notebook adds RL training on top of our 4 heuristic experiments:

```
Exp 1  CTA Baseline Validation   ─ reproduces paper Table I CTA row
Exp 2  Random vs CTA             ─ shows heuristic value (lower bound)
Exp 3  CTA Scalability           ─ original contribution (5 sizes)
Exp 4  Queue Sensitivity         ─ original contribution (6 queue depths)
This   IAC Training              ─ real RL, comparable to paper Table I IAC row
```

Together: Random < CTA (rule-based) < IAC (neural RL) — the full story of
why learned policies are better than hand-crafted heuristics.